# RAG 기반 AI 응용서비스 설계 2일 코스

## 1일차 목차

| 번호 | 주제 | 핵심 내용 |
|---|---|---|
| 1 | 환경 설정 | 필수 라이브러리 설치, OpenAI API 키 설정 |
| 2 | LangChain 소개 | LangChain의 역할과 LCEL 개요 |
| 3 | 프롬프트(Prompt) | System / User / Assistant 프롬프트, `PromptTemplate`, Few-shot Learning |
| 4 | Q&A 챗봇 실습 (LCEL) | LCEL 기본 체인 구성, `invoke` / `ainvoke` / `batch` / `stream` 비교 |
| 5 | RAG 개념 이해 | LLM 한계, Retrieval+Generation 결합, RAG 구성 요소, 벡터 검색 원리 |
| 6 | RAG 파이프라인 구성 실습 | PDF 로드, 청킹, 임베딩/벡터 DB 저장, 검색/답변 생성 |
| 7 | 기본형 RAG의 한계 | Naive RAG 개념, Indexing/Retrieval/Generation 한계 |
| 8 | Advanced RAG 입문 | 고급 Chunking, Query Rewriting |

## 1. 환경 설정

### 1.1 필수 라이브러리 설치

이번 실습은 PDF 로딩, 임베딩, 벡터 DB 저장, OpenAI 호출이 포함되지만  
**로컬 환경, 클라우드 환경, Colab 어디서든 실행할 수 있습니다.**

다만 PDF 파일 경로, `.env` 설정, 벡터 DB 저장 위치를 계속 유지해야 하므로  
**한 번 정한 실행 환경을 실습 끝까지 일관되게 사용하는 것**을 권장합니다.
Colab을 사용할 경우에는 PDF 업로드 또는 Drive 마운트, API Key 설정 방식을 그 환경에 맞게 조정하면 됩니다.

설치 명령은 아래와 같습니다.

```bash
pip install -r requirements.txt
```

설치 후에는 커널이나 파이썬 세션을 한 번 다시 시작하면 오류를 줄일 수 있습니다.
        

### 1.2 참고: 라이브러리별 역할 설명

| 라이브러리 | 역할 |
|---|---|
| `langchain` | 프롬프트, 체인, 출력 파서 같은 기본 구성 |
| `langchain-openai` | OpenAI 모델과 임베딩 연결 |
| `langchain-community` | PDF 로더, FAISS 같은 실습용 도구 |
| `langchain-text-splitters` | 문서를 청크로 나누는 기능 |
| `faiss-cpu` | 벡터 검색용 로컬 DB |
| `python-dotenv` | `.env` 파일에서 API Key 불러오기 |

이 절에서는 각 라이브러리를 "문서 읽기 -> 나누기 -> 임베딩 -> 저장 -> 검색" 흐름을 구성하는 도구로 보고 역할을 정리합니다.
        

### 1.3 OpenAI API 키 설정 (.env 파일, 보안 주의)

**API란?**

하나의 프로그램이 다른 프로그램이나 시스템과 **데이터를 주고받고 기능을 활용할 수 있도록 연결해주는 인터페이스**를 의미합니다.  
즉, **서로 다른 소프트웨어 사이의 통신 규칙**이라고 이해하면 됩니다.

예를 들어 OpenAI의 **GPT API**는 사용자가 직접 모델을 설치하지 않아도,  
OpenAI 서버에 **질문(요청)** 을 보내고 그 결과로 **답변(응답)** 을 받을 수 있게 해줍니다.

→ 쉽게 말해, **필요할 때마다 GPT에게 요청을 보내 답변을 받아오는 온라인 시스템**입니다.

이 API는 **사용량 기반으로 과금**되며, ChatGPT Plus/Pro 같은 웹앱 구독과는 **별도 요금 체계**로 운영됩니다.  
즉, ChatGPT를 결제하고 있더라도 API 사용량은 따로 계산됩니다.

※ 현재 사용 가능한 대표 GPT API 모델 일부 (2026년 3월 27일 기준, text token standard price)

| 모델 이름 | 설명 | 입력 가격 (per 1M tokens) | 출력 가격 (per 1M tokens) | 추천 용도 |
|-----------|--------|---------------------------:|----------------------------:|-----------|
| `gpt-3.5-turbo` | 초창기 레거시 모델 | **$0.50** | **$1.50** | 기존 예제 호환, 구형 자료 참고용 |
| `gpt-4o-mini` | 빠르고 저렴한 범용 소형 모델 | **$0.15** | **$0.60** | 가벼운 멀티모달 앱, FAQ 챗봇, 분류/추출, 파인튜닝용 |
| `gpt-4.1-mini` | 작고 빠른 GPT-4.1 계열 모델 | **$0.40** | **$1.60** | 지시 이행, 툴 호출, 긴 문맥 실무형 API 앱 |
| `gpt-4.1-nano` | 더 가볍고 저렴한 초경량 모델 | **$0.10** | **$0.40** | 초저비용 분류, 간단한 자동화, 대량 처리 |
| `o4-mini` | 빠른 추론에 강한 모델 | **$1.10** | **$4.40** | 복잡한 추론, 코딩 보조, 시각 이해 |
| `gpt-5-mini` | GPT-5 계열 비용 효율형 모델 | **$0.25** | **$2.00** | 저지연·고볼륨 실서비스, 명확한 업무형 프롬프트 |
| `gpt-5-nano` | GPT-5 계열 최저가 모델 | **$0.05** | **$0.40** | 요약, 분류, 태깅, 대량 배치 처리, 입문 예제 |

※ 위 표는 대표 모델 일부만 정리한 것이며, `cached input`, `Batch API` 할인 등은 제외했습니다.  
최신 목록과 상세 요금은 공식 문서에서 확인하세요:
- https://developers.openai.com/api/docs/models
- https://openai.com/api/pricing/

**API Key란?**  
GPT API를 사용하려면 OpenAI 서버가 요청을 보낸 사용자를 식별하고, 해당 사용량과 과금 내역을 추적할 수 있어야 합니다.  
이를 위해 OpenAI는 각 사용자에게 **고유한 인증 토큰(API Key)** 을 발급합니다.

이 키는 **요금 청구와 권한 식별의 기준**이 되는 민감한 정보이므로, 외부에 노출되면 다른 사람이 임의로 사용해 비용이 발생할 수 있습니다.

**발급 절차**

1. [https://platform.openai.com/](https://platform.openai.com/) 접속  
2. 로그인 또는 계정 신규 생성  
3. 첫 API Key 발급 시에는 **Billing 설정에서 결제 수단 또는 prepaid credits 설정이 필요**  
4. API key 페이지에서 `Create new secret key` 클릭
6. 발급된 Key를 바로 복사해 안전한 위치에 보관  

**발급된 Key는 한 번만 표시되므로**, 즉시 `.env` 파일이나 안전한 비밀 저장소에 백업해 두는 것이 좋습니다.

**API Key 관리 원칙**

- 코드에 직접 쓰지 않기
- `.env` 파일에 저장하기
- GitHub나 메신저에 공유하지 않기
- 실습 후에도 외부 업로드 여부를 한 번 더 확인하기
- 사용량이 걱정되면 Billing / Usage Limits에서 예산과 한도를 함께 확인하기
        

실습 폴더 루트에 `.env` 파일을 만들고 아래처럼 저장합니다.

```bash
OPENAI_API_KEY=sk-xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx
```

이후에는 `python-dotenv`로 불러와 사용합니다.

```python
from dotenv import load_dotenv
import os

load_dotenv()
```

#### 1.3.1 주의
`.env` 파일은 절대 GitHub나 외부 저장소에 업로드하지 마세요.  
API Key가 노출되면 타인이 임의로 호출해 과금이 발생할 수 있습니다.
        

## 2. LangChain 소개

LangChain은 **LLM 활용 파이프라인을 코드로 연결해주는 프레임워크**입니다.

예를 들어,

- 프롬프트 만들기
- 문서 검색기 연결하기
- 모델 호출하기
- 출력 형식 정리하기

같은 단계를 하나의 흐름으로 연결할 수 있게 도와줍니다.
        

<img src="image/LangChain.png" width="560">

LangChain을 처음 배울 때는 "모델 호출 라이브러리"라기보다,  
**프롬프트 -> 검색 -> 모델 -> 출력 정리**를 하나의 체인으로 묶어주는 프레임워크라고 이해하면 훨씬 쉽습니다.


실무에서는 모델만 단독으로 쓰기보다,
**입력 정리 -> 검색 -> 생성 -> 후처리**를 함께 설계해야 할 때가 많습니다.

LangChain은 이런 여러 단계를 코드로 연결하기 쉽게 만들어줍니다.
즉, LangChain은 "모델 사용"을 "파이프라인 흐름 설계"로 확장하는 도구입니다.
        

<img src="image/chatgpt.jpg" width="560">

겉으로는 챗봇이 한 번에 답하는 것처럼 보여도, 실제 서비스는 **입력 해석 -> 관련 정보 탐색 -> 답변 구성** 같은 여러 단계를 거칩니다.  
이런 숨은 흐름을 코드로 구조화하는 데 LangChain이 자주 사용됩니다.


또한 LangChain은 **LCEL (LangChain Expression Language)** 로  
각 단계를 `|` 연산자로 자연스럽게 연결할 수 있습니다.

예:

```text
질문 -> 프롬프트 -> LLM -> 출력 파서
```

오늘 실습에서는 이 LCEL 문법을 이용해 RAG 체인을 구성합니다.
        

### 2.1 간단한 예시 코드

In [2]:
import warnings  # 경고를 숨길 때 쓰는 표준 라이브러리입니다.
warnings.filterwarnings("ignore")  # 실습과 무관한 경고는 모두 숨깁니다.

from langchain_openai import ChatOpenAI  # OpenAI 채팅 모델을 LangChain에서 호출합니다.
from langchain_core.prompts import PromptTemplate  # 문자열 프롬프트 템플릿을 만듭니다.
from langchain_core.output_parsers import StrOutputParser  # 모델 응답에서 텍스트만 꺼냅니다.

import os
from dotenv import load_dotenv
load_dotenv(override=True)

True

In [3]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
prompt = PromptTemplate.from_template("'{topic}'를 한 문장으로 설명해줘.")  # topic 값만 바꿔 재사용할 프롬프트입니다.
chain = prompt | llm | StrOutputParser()  # 프롬프트, 모델, 출력 파서를 하나의 흐름으로 묶습니다.

chain.invoke({"topic": "LangChain"})  # topic 값을 넣어 최종 응답을 확인합니다.

'LangChain은 다양한 언어 모델과 데이터 소스를 통합하여 자연어 처리 애플리케이션을 구축하고 관리할 수 있도록 돕는 프레임워크입니다.'

## 3. 프롬프트(Prompt)

프롬프트는 모델에게 **무엇을, 어떤 방식으로 답할지** 알려주는 입력입니다.

같은 질문이라도 프롬프트가 다르면 답변의 수준과 형식이 달라집니다.

- "고양이를 설명해줘"
- "고양이를 5살 아이도 이해하게 설명해줘"

처럼 요구를 구체적으로 쓸수록 원하는 출력에 가까워집니다.
        

### 3.1 LLM은 세 가지 종류의 프롬프트로 대화한다

일반 ChatGPT에서 사람들이 흔히 말하는 `프롬프트`는 보통 **지금 입력한 질문 한 줄**입니다.  
하지만 LLM 앱을 설계할 때는, **한 번의 턴을 보낼 때도 역할이 다른 메시지들을 함께 묶어** 모델에 전달합니다.

즉, 앱 입장에서는 사용자 질문만 보내는 것이 아니라,
- 어떻게 답해야 하는지
- 이번에 무엇을 물었는지
- 바로 직전에 어떤 대화가 있었는지
를 나눠서 함께 보내는 구조입니다.

왜 이렇게 나누어 보내느냐 하면, 이 세 가지를 한 문장에 섞어 쓰면 **무엇이 규칙이고, 무엇이 이번 요청이고, 무엇이 이전 맥락인지** 구분이 흐려지기 쉽기 때문입니다.  
역할을 나눠 두면 모델도 **항상 지킬 규칙**, **지금 처리할 질문**, **이전 대화 흐름**을 더 분명하게 받아들일 수 있어 답변이 더 일관되고, 대화가 길어져도 관리하기 쉬워집니다.

| 개념 설명용 이름 | LangChain에서 자주 보이는 이름 | 역할 |
|---|---|---|
| **System** | `system` / `SystemMessage` | 모델의 역할과 규칙 지정 |
| **User** | `user` / `human` / `HumanMessage` | 사용자가 한 질문이나 요청 |
| **Assistant** | `assistant` / `ai` / `AIMessage` | 이전 답변 맥락 |

이 수업에서는 먼저 **개념 설명은 `System / User / Assistant` 기준**으로 설명합니다.

- **System**: 답변 방식과 규칙을 정함
- **User (= Human)**: 사용자가 한 질문이나 요청을 담음
- **Assistant (= AI)**: 이전에 어떤 답을 했는지 이어줌

뒤에서 볼 `ChatPromptTemplate`은 이런 메시지들을 한 번에 묶어 모델에 보내기 위한 도구입니다.




### 3.2 자주 나오는 질문: 왜 `user` 대신 `human`이라고도 쓰나요?

OpenAI 계열 Chat API는 보통 역할(role)을 `system / user / assistant`로 설명합니다.  
반면 LangChain은 내부 메시지 클래스를 `SystemMessage / HumanMessage / AIMessage`로 자주 설명합니다.

그래서 `ChatPromptTemplate.from_messages([...])` 예시를 보면
- 어떤 자료는 `("user", "...")`
- 어떤 자료는 `("human", "...")`
처럼 보일 수 있습니다.

핵심은 **둘 다 사람 쪽 입력 메시지를 가리킨다**는 점입니다.  
이 교안에서는 혼란을 줄이기 위해 **가능하면 `user`로 통일**하되, 다른 자료에서 `human`을 보더라도 같은 의미라고 이해하면 됩니다.


### 3.3 예시 1. `PromptTemplate`로 한 문장 프롬프트 만들기

LangChain에서는 프롬프트를 문자열로 매번 직접 쓰기보다,
`PromptTemplate` 으로 **재사용 가능한 틀**을 만들어 씁니다.

즉, 고정 문장 안에 `{topic}` 같은 변수를 넣고
상황에 따라 값만 바꿔 쓰는 방식입니다.

가장 단순한 형태의 프롬프트 예시라고 보면 됩니다.
바로 다음 예시 2에서는 이 개념을 확장해, 여러 메시지를 묶는 `ChatPromptTemplate`를 살펴봅니다.


In [ ]:
from langchain_core.prompts  import PromptTemplate  # 문자열 프롬프트 템플릿을 만드는 도구입니다.

# {topic} 부분에 사용자의 입력이 들어가도록 템플릿을 만듭니다.
prompt = PromptTemplate.from_template("'{topic}'에 대해 초보자도 이해할 수 있게 한 문장으로 설명해줘.")

# format()은 실제 값을 넣어 최종 프롬프트 문장을 확인할 때 씁니다.
prompt.format(topic="LangChain")
# 결과 → "'LangChain'에 대해 초보자도 이해할 수 있게 한 문장으로 설명해줘."


### 3.4 예시 2. `ChatPromptTemplate`로 대화 흐름 만들기

`PromptTemplate`이 한 문장짜리 프롬프트 틀을 만드는 도구였다면,
`ChatPromptTemplate`은 **역할이 다른 여러 메시지를 묶어 하나의 대화 맥락**을 만드는 도구입니다.

이 예시에서 볼 점은 세 가지입니다.

- `system` 메시지로 모델의 역할과 답변 스타일을 먼저 정합니다.
- `user`와 `assistant` 메시지를 함께 넣어, 이전에 어떤 대화가 오갔는지 전달합니다.
- 마지막 `user` 질문에 대해 모델이 **앞선 대화를 이어서 답하도록** 만듭니다.

즉, 챗봇처럼 여러 차례 오가는 대화를 다루고 싶을 때는
문자열 프롬프트 하나보다 `ChatPromptTemplate`이 더 자연스럽습니다.

이번 예시는 변수 치환보다 **메시지 구조 자체를 이해하는 것**이 핵심이라
`chain.invoke({})`처럼 빈 입력으로 실행합니다.


In [ ]:
from langchain_openai import ChatOpenAI  # OpenAI 채팅 모델을 LangChain에서 호출합니다.
from langchain_core.prompts  import ChatPromptTemplate  # 여러 메시지 역할을 묶는 채팅 프롬프트 템플릿입니다.
from langchain_core.output_parsers import StrOutputParser  # 모델 응답 객체에서 텍스트만 꺼내는 파서입니다.

# 1. 모델을 준비합니다.
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

# 2. 역할이 다른 메시지를 순서대로 묶어 대화 맥락을 만듭니다.
# - System: 모델의 기본 역할(여행 전문가)
# - User (= Human): 사용자의 첫 질문 (가을 여행지 추천)
# - Assistant: 직전에 모델이 대답했던 내용 (AI의 응답)
# - User (= Human): 현재 사용자의 새로운 질문 (부산 일정표 요청)
# 참고: LangChain 자료에서는 같은 사람 입력 메시지를 ("user", ...) 또는 ("human", ...)로 적는 예시가 섞여 있습니다.
# 이 교안에서는 OpenAI 역할명과 맞추기 위해 user로 통일합니다.
prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 친절한 여행 전문가야. 사용자의 취향에 맞게 국내 여행지를 추천하고, 일정도 구체적으로 제안해줘."),
    ("user", "가을에 2박 3일로 갈만한 국내 여행지를 추천해줘."),
    ("assistant", "부산, 여수, 속초가 좋을 것 같아요! 각각 바다와 음식, 분위기가 다르니 선택에 따라 즐길 거리가 달라요."),
    ("user", "그럼 이번엔 부산으로 2박 3일 일정표를 만들어줘.")
])

parser = StrOutputParser()  # 응답 객체에서 최종 답변 텍스트만 꺼냅니다.
chain = prompt | llm | parser  # 프롬프트 → 모델 → 텍스트 출력 흐름으로 연결합니다.

result = chain.invoke({})  # 변수 없이 고정 대화 맥락만으로 응답을 생성합니다.
print(result)


### 3.5 예시 3. 퓨샷 러닝(Few-shot Learning)

**Few-shot Learning**은 모델에게 원하는 출력 패턴의 예시를 1~2개 보여주고,
새 입력에도 같은 형식으로 답하게 만드는 방법입니다.

핵심은 간단합니다.

- 설명만 하는 것보다 **예시를 함께 보여주면** 출력 형식이 더 안정됩니다.
- RAG에서도 답변 톤, 요약 형식, 인용 방식뿐 아니라
  **구어체 질문을 문서 검색형 표현으로 바꾸는 작업**에 활용할 수 있습니다.
- 다만 예시가 많아질수록 토큰을 더 쓰므로, 실무에서는 **짧고 대표적인 예시**만 넣는 편이 좋습니다.
        

In [ ]:
### 코드 예시: Few-shot으로 충청도 사투리 표준어 변환하기
from langchain_openai import ChatOpenAI  # OpenAI 채팅 모델을 LangChain에서 호출합니다.
from langchain_core.prompts import ChatPromptTemplate  # 예시 대화를 포함한 채팅 프롬프트를 만듭니다.
from langchain_core.output_parsers import StrOutputParser  # 모델 응답 객체에서 텍스트만 꺼내는 파서입니다.

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)

# 충청도 사투리 예시 질문/답변을 함께 넣어 원하는 출력 형식을 학습시킵니다.
prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 사용자의 충청도 사투리 표현을 자연스러운 표준어로 풀어 쓰는 도우미다. 항상 '원문 의미'와 '표준어 표현' 두 줄로 답한다."),
    ("user", "점심은 묵었슈?"),
    ("assistant", "원문 의미: 점심을 먹었는지 묻고 있다.\n표준어 표현: 점심은 먹었어요?"),
    ("user", "그 일은 오늘 안에 혀야 쓰겄는디?"),
    ("assistant", "원문 의미: 그 일을 오늘 안에 해야 하는지 확인하고 있다.\n표준어 표현: 그 일은 오늘 안에 해야 하나요?"),
    ("user", "{question}")
])

chain = prompt | llm | StrOutputParser()  # few-shot 프롬프트와 모델을 하나의 체인으로 묶습니다.
result = chain.invoke({"question": "거기까진 얼마나 걸리는 겨?"})  # 새 질문도 같은 형식으로 바뀌는지 확인합니다.
print(result)


## 4. Q&A 챗봇 실습 (LCEL)

이 절의 목표는 아주 단순합니다.

**질문을 입력받아 일정한 형식으로 답하는 작은 챗봇**을 만들어보는 것입니다.

여기서는 복잡한 기능보다 아래 흐름에 익숙해지는 것이 중요합니다.

1. 프롬프트 만들기
2. LLM 연결하기
3. 출력 파서 붙이기
4. LCEL로 하나의 체인으로 연결하기
        

### 실습 문제 1: LCEL Q&A 체인 완성하기

아래 셀은 LCEL 체인을 직접 완성하는 문제입니다.

채울 값은 두 가지입니다.

- `system_message`: 답변 역할과 형식을 정하는 문장
- `question`: 실제로 모델에게 물어볼 질문

답변에는 아래 세 가지가 드러나도록 프롬프트를 설계해 봅니다.

- **정의**: 무엇인지
- **이유**: 왜 중요한지
- **예시**: 쉬운 비유나 사례

In [ ]:
### 아래 가이드를 참고해 직접 완성해보세요

from langchain_openai import ChatOpenAI  # OpenAI 채팅 모델을 LangChain에서 호출합니다.
from langchain_core.prompts import ChatPromptTemplate  # 여러 메시지 역할을 묶는 채팅 프롬프트 템플릿입니다.
from langchain_core.output_parsers import StrOutputParser  # 모델 응답 객체에서 텍스트만 꺼내는 파서입니다.

# 1. 모델을 선언하세요.
# 조건: model은 "gpt-4o-mini"처럼 가벼운 모델을 사용하고, temperature=0.7로 설정하세요.
llm = ChatOpenAI(...)

# 2. 프롬프트를 구성하세요.
# - ChatPromptTemplate.from_messages([...])를 사용하세요.
# - system에는 "기술 개념을 쉽게 설명하는 AI 선생님" 역할을 넣으세요.
# - user에는 {question} 변수를 사용하세요.
# - 다른 자료에서 human이 보여도 같은 사람 입력 역할입니다.
prompt = ChatPromptTemplate.from_messages([...])

# 3. 출력 파서를 선언하세요.
parser = None

# 4. LCEL 체인을 완성하세요.
chain = None

# 5. 테스트 질문을 넣고 실행하세요.
question = "RAG는 어떤 구조야?"
answer = chain.invoke(None)
print(answer)

# 6. 답변에 정의 / 중요성 / 예시가 잘 들어갔는지 확인한 뒤 프롬프트를 수정해보세요.


<details>
<summary>정답 보기</summary>

```python 
from langchain_openai import ChatOpenAI
from langchain_core.prompts  import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 모델 선언 (이런 예시는 gpt-4o-mini로 충분합니다.)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

# 프롬프트 설계
# system: 챗봇의 역할 정의
# user: {question} 변수를 통해 질문 전달
# 참고: 다른 자료에서 ("human", "{question}")를 보더라도 같은 의미입니다.
prompt = ChatPromptTemplate.from_messages([
    ("system", 
     "너는 기술 개념을 알기 쉽게 설명하는 AI 선생님이야. "
     "답변은 반드시 1. 정의 2. 이유(중요성) 3. 쉬운 예시를 포함해야 해. "
     "필요하다면 비유를 사용해도 좋아."),
    ("user", "{question}")
])

# 출력 파서
parser = StrOutputParser()

# LCEL 체인 구성
chain = prompt | llm | parser

# 실행 테스트
question = "REST API란?"
answer = chain.invoke({"question": question})

print(f"🧠 질문: {question}\n")
print(f"💬 답변:\n{answer}")
```
</details>

### 4.1 체인 실행 메서드 비교 실험: `invoke`, `ainvoke`, `batch`, `stream`

기본 LCEL 체인을 한 번 만들어봤으니, 이제는 **같은 chain을 어떤 방식으로 실행할 수 있는지** 비교해보겠습니다.  
문법만 외우기보다 **같은 입력 10개를 실제로 돌려보며 시간 차이와 사용 목적을 비교**하는 편이 훨씬 이해가 빠릅니다.

| 메서드 | 비교 포인트 | 이런 상황에 적합 |
|---|---|---|
| `invoke()` | 가장 단순한 순차 처리 기준선 | 입력이 1개이거나 디버깅이 우선일 때 |
| `ainvoke()` | 요청 1개를 비동기로 실행. 여러 개를 직접 스케줄링하면 동시에 진행 가능 | 웹 서버, 다중 사용자 요청, I/O 대기가 많을 때 |
| `batch()` | 여러 입력을 한 번에 넘기는 일괄 처리 | 평가셋 실행, 대량 생성, 반복 실험 |
| `stream()` | 전체 완료 시간보다 **첫 텍스트가 언제 보이기 시작하는지** | 채팅 UI처럼 사용자가 기다리는 화면 |

`ainvoke()` 하나만 쓰면 요청 1개를 비동기로 실행합니다.
여러 요청을 동시에 진행하려면, 아래 코드처럼 `ainvoke()`를 여러 번 호출해 태스크를 만든 뒤 한꺼번에 기다리면 됩니다.

이때 모양만 보면 `abatch()`와 비슷해 보일 수 있지만, 의도는 다릅니다.
- `ainvoke()` 예시는 **개별 요청을 직접 만들고 스케줄링하는 방식**
- `batch()` 예시는 **입력 리스트를 한 번에 넘기는 방식**

이 비교 실험에서는 **`gpt-4o-mini`** 를 사용합니다.
핵심은 모델 특성보다 `invoke / ainvoke / batch / stream` 실행 방식 차이를 보는 것입니다.

주의: 실제 시간 차이는 네트워크 상태, 모델 혼잡도, rate limit에 따라 달라질 수 있습니다.  
그래서 이 실습은 **절대 속도**보다 `순차 처리 vs 동시 처리 vs 일괄 처리 vs 체감 응답성`의 차이를 이해하는 데 목적이 있습니다.



In [ ]:
import asyncio  # 비동기 요청을 동시에 실행할 때 쓰는 표준 라이브러리입니다.
import statistics  # 실행 결과의 평균값을 계산할 때 씁니다.
import time  # 실행 시간을 재는 표준 라이브러리입니다.

from dotenv import load_dotenv  # .env 파일의 환경 변수를 현재 세션으로 불러옵니다.
from langchain_core.output_parsers import StrOutputParser  # 모델 응답 객체에서 텍스트만 꺼내는 파서입니다.
from langchain_core.prompts import PromptTemplate  # 문자열 프롬프트 템플릿을 만드는 도구입니다.
from langchain_openai import ChatOpenAI  # OpenAI 채팅 모델을 LangChain에서 호출합니다.

load_dotenv(override=True)  # 실습에 필요한 API 키를 현재 세션에 반영합니다.

# 비교 실험용 모델
benchmark_llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
)
stream_llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    streaming=True,
)

prompt = PromptTemplate.from_template("'{topic}'를 초보자도 이해할 수 있게 한 문장으로 설명해줘.")  # 같은 질문 형식을 계속 재사용합니다.
parser = StrOutputParser()  # 모델 응답에서 텍스트만 남겨 비교를 단순하게 만듭니다.

# 같은 프롬프트를 실행 방식만 바꿔 비교할 수 있도록 체인을 준비합니다.
chain = prompt | benchmark_llm | parser
stream_chain = prompt | stream_llm | parser

# 속도 비교를 위해 입력 수와 난이도를 비슷하게 맞춘 주제 목록입니다.
topics = [
    "LangChain",
    "PromptTemplate",
    "LCEL",
    "RAG",
    "Embedding",
    "Vector DB",
    "Chunking",
    "Retriever",
    "Query Rewriting",
    "Hallucination",
]
benchmark_inputs = [{"topic": topic} for topic in topics]  # 각 실행 메서드에 같은 입력을 넣습니다.

print("비교 대상 입력 수:", len(benchmark_inputs))
print("비교용 topics:", topics)


In [ ]:
# 1. invoke(): 순차 처리의 기준선을 만들기 위해 요청을 하나씩 실행합니다.
invoke_outputs = []
invoke_start = time.perf_counter()  # 전체 순차 실행 시작 시점을 기록합니다.

for payload in benchmark_inputs:
    invoke_outputs.append(chain.invoke(payload))  # 입력 1개가 끝난 뒤 다음 입력을 실행합니다.

invoke_total = time.perf_counter() - invoke_start
invoke_avg = invoke_total / len(benchmark_inputs)  # 총 시간을 입력 수로 나눠 평균을 계산합니다.

print("=== invoke benchmark ===")
print(f"총 소요 시간: {invoke_total:.2f}초")
print(f"요청 1개당 평균: {invoke_avg:.2f}초")
print("샘플 응답 2개:")
for i, text in enumerate(invoke_outputs[:2], 1):
    print(f"{i}. {text}")


In [ ]:
# 2. ainvoke(): 개별 요청을 비동기로 만들고, 마지막에 한꺼번에 기다립니다.
async def run_ainvoke_benchmark():  # ainvoke를 동시에 실행해 총 시간을 잽니다.
    ainvoke_start = time.perf_counter()  # 비동기 전체 실행 시작 시점을 기록합니다.

    tasks = []  # 동시에 기다릴 비동기 작업을 모읍니다.
    for payload in benchmark_inputs:
        # await 없이 태스크만 먼저 만들어 두면, 여러 요청이 동시에 진행될 준비를 합니다.
        tasks.append(asyncio.create_task(chain.ainvoke(payload)))

    # 여기서 모든 태스크가 끝날 때까지 한꺼번에 기다립니다.
    results = await asyncio.gather(*tasks)  # 모든 요청이 끝날 때까지 한 번에 기다립니다.
    ainvoke_total = time.perf_counter() - ainvoke_start
    return results, ainvoke_total

ainvoke_outputs, ainvoke_total = await run_ainvoke_benchmark()  # 비동기 실행 결과와 총 시간을 받습니다.
ainvoke_avg = ainvoke_total / len(benchmark_inputs)  # 총 시간을 입력 수로 나눠 비교용 평균을 만듭니다.

print("=== ainvoke benchmark ===")
print(f"총 소요 시간: {ainvoke_total:.2f}초")
print(f"요청 1개당 평균(총 시간 기준): {ainvoke_avg:.2f}초")
print("샘플 응답 2개:")
for i, text in enumerate(ainvoke_outputs[:2], 1):
    print(f"{i}. {text}")

In [ ]:
# 3. batch(): 여러 입력을 한 번에 넘겨 일괄 처리 시간을 확인합니다.
batch_start = time.perf_counter()  # batch 전체 실행 시작 시점을 기록합니다.
batch_outputs = chain.batch(benchmark_inputs)  # 같은 입력 리스트를 한 번에 전달합니다.
batch_total = time.perf_counter() - batch_start
batch_avg = batch_total / len(benchmark_inputs)  # 총 시간을 입력 수로 나눠 비교용 평균을 만듭니다.

print("=== batch benchmark ===")
print(f"총 소요 시간: {batch_total:.2f}초")
print(f"요청 1개당 평균(총 시간 기준): {batch_avg:.2f}초")
print("샘플 응답 2개:")
for i, text in enumerate(batch_outputs[:2], 1):
    print(f"{i}. {text}")


In [ ]:
# 4. stream(): 첫 번째 요청은 실시간으로 출력해 보고, 전체 benchmark도 함께 측정합니다.
stream_outputs = []
stream_first_token_times = []
stream_total_start = time.perf_counter()  # 스트리밍 전체 실험 시작 시점을 기록합니다.

print("=== stream demo (첫 번째 요청 실시간 출력) ===")

for idx, payload in enumerate(benchmark_inputs):
    chunks = []
    request_start = time.perf_counter()  # 각 요청의 시작 시점을 따로 기록합니다.
    first_chunk_time = None

    if idx == 0:
        print(f"topic: {payload['topic']}")
        print("응답 조각:", end=" ", flush=True)

    for chunk in stream_chain.stream(payload):
        chunk_text = str(chunk)
        if not chunk_text:
            continue  # 빈 이벤트는 건너뛰고, 실제 텍스트가 보일 때부터 측정합니다.

        if first_chunk_time is None:
            first_chunk_time = time.perf_counter() - request_start  # 첫 번째 텍스트 조각이 보이기까지 걸린 시간입니다.
        chunks.append(chunk_text)  # 잘린 응답 조각을 순서대로 모읍니다.
        if idx == 0:
            print(chunk_text, end="", flush=True)  # 첫 번째 요청은 조각이 도착하는 즉시 화면에 보여줍니다.

    final_text = "".join(chunks)
    stream_outputs.append(final_text)  # 조각을 이어 붙여 최종 응답을 만듭니다.
    stream_first_token_times.append(first_chunk_time or 0.0)

    if idx == 0:
        print("\n")
        print(f"(첫 텍스트 조각 도착: {stream_first_token_times[-1]:.2f}초)")
        print()

stream_total = time.perf_counter() - stream_total_start
stream_avg_total = stream_total / len(benchmark_inputs)  # 요청당 평균 완료 시간을 계산합니다.
stream_avg_ttft = statistics.mean(stream_first_token_times)  # TTFT 평균을 계산합니다.

print("=== stream benchmark ===")
print(f"10개 전체 완료 시간: {stream_total:.2f}초")
print(f"요청 1개당 평균 완료 시간: {stream_avg_total:.2f}초")
print(f"평균 첫 텍스트 조각 도착 시간: {stream_avg_ttft:.2f}초")
print("첫 번째 요청의 최종 응답(다시 확인):")
print(stream_outputs[0])


In [ ]:
# 네 가지 실행 방식의 결과를 한 줄씩 모아 비교합니다.
print("=== 실행 방식 비교 요약 ===")
print(f"invoke  : total={invoke_total:.2f}s | avg={invoke_avg:.2f}s")
print(f"ainvoke : total={ainvoke_total:.2f}s | avg={ainvoke_avg:.2f}s")
print(f"batch   : total={batch_total:.2f}s | avg={batch_avg:.2f}s")
print(f"stream  : total={stream_total:.2f}s | avg_complete={stream_avg_total:.2f}s | avg_ttft={stream_avg_ttft:.2f}s")

print("\n해석 포인트")
print("1. invoke는 가장 직관적인 기준선입니다.")
print("2. ainvoke는 여러 요청을 동시에 처리할 때 throughput 향상을 기대할 수 있습니다.")
print("3. batch는 평가셋/대량 처리에서 편리하지만, 내부 동작은 모델 제공자 구현에 따라 차이가 있습니다.")
print("4. stream은 총 완료 시간보다 사용자가 언제 답을 보기 시작하고, 답이 어떻게 조금씩 이어지는지가 더 중요할 때 의미가 큽니다.")
print("5. rate limit이나 네트워크 상태에 따라 결과는 매번 조금씩 달라질 수 있습니다.")


#### 4.1.1 관찰 질문

1. 방금 만든 LCEL 체인을 서비스에 붙인다면, 어떤 실행 방식이 가장 먼저 떠오르나요?
2. 이번 실행에서는 `ainvoke()`와 `batch()` 중 어떤 방식이 더 빨랐나요?
3. `stream()`은 총 완료 시간보다 왜 `평균 첫 텍스트 조각 도착 시간`이 더 중요한가요?

## 5. RAG 개념 이해

### 5.1 문제 정의: LLM만으로는 왜 부족한가?

#### 5.1.1 LLM의 한계

LLM은 매우 강력하지만, 실무형 문서 질의응답에는 한계가 있습니다.

- **환각**: 틀린 정보를 그럴듯하게 말할 수 있음
- **최신성 부족**: 학습 이후 바뀐 정보를 모를 수 있음
- **희소 정보 취약**: 특정 회사 문서나 매뉴얼 같은 정보에 약함
- **근거 부족**: 왜 그렇게 답했는지 출처를 보여주기 어려움

이러한 한계를 보완하려면, LLM이 자체 지식에만 의존하지 않고 질문과 관련된 외부 정보를 함께 활용할 수 있어야 합니다.

#### 5.1.2 검색(Retrieval)과 생성(Generation) 결합 필요성

이때 필요한 것이 바로 **검색(Retrieval)** 과 **생성(Generation)** 의 결합입니다.

1. **검색(Retrieval)**: 질문과 관련된 문서나 정보를 찾는 단계
2. **생성(Generation)**: 찾은 내용을 바탕으로 답변을 만드는 단계

즉, 먼저 근거가 될 정보를 찾고, 그다음 그 내용을 바탕으로 답변을 구성하는 구조가 필요합니다.
        

##### 1) 웹 검색(Web Retrieval)

- 최신 뉴스나 변경된 정보에 강함
- 실시간 정보가 필요할 때 유용함
- 예: 최근 정책, 최신 제품, 오늘 일정
        

##### 2) 문서 검색(Document Retrieval)

- PDF, 매뉴얼, 규정집, 내부 문서에 강함
- 외부에 잘 알려지지 않은 정보를 찾을 때 유용함
- 예: 회사 규정, 장비 매뉴얼, 투자 안내서
        

##### 3) 생성(Generation)

생성은 검색된 내용을 그대로 붙여넣는 것이 아니라, 질문에 맞게 정리해 자연스러운 답변으로 만드는 단계입니다.

> 즉, **검색은 근거 확보**, **생성은 답변 구성** 역할을 맡습니다.

이 둘을 결합한 구조가 바로 **RAG**입니다.
        

##### 예시 및 정리

예를 들어 `사내 복무 규정`, `출장비 정산 지침`, `보안 정책` 문서를 넣어 만든 사내 Q&A 챗봇이 있다고 가정해봅시다.  
이때 사용자가  
"배우자 출산휴가는 며칠까지 사용할 수 있어?"  
라고 물었다고 가정해봅시다.

- **LLM만 사용**: 출산휴가 제도가 있을 것이라는 일반 설명은 할 수 있지만, 회사 규정에 적힌 정확한 일수와 조건은 맞추기 어려움
- **RAG 사용**: 저장된 규정 문서들 중 질문과 관련된 부분을 찾아, 실제 문서에 적힌 휴가 일수와 조건을 근거로 답할 수 있음

즉, RAG의 핵심은 단순히 그럴듯하게 답하는 것이 아니라,  
**문서에 실제로 있는 정보를 기반으로 정확하게 답하게 만드는 것**입니다.

#### 5.1.3 RAG 구성 요소

RAG는 아래 세 요소로 이해하면 됩니다.

- **Retriever**: 관련 문서를 찾음
- **Augmentation**: 찾은 문서를 프롬프트에 넣음
- **Generator**: 그 문서를 바탕으로 답변함
        

##### RAG 구조 도식

<img src="image/RAG.png" width="650">

위 그림처럼 RAG는 보통 `문서 준비 -> 청킹/임베딩 -> 벡터 DB 저장 -> 질문 검색 -> 답변 생성` 흐름으로 이해하면 됩니다.  
즉, **질문과 관련된 근거를 먼저 찾고 그다음 답을 생성하는 구조**가 핵심입니다.


#### 5.1.4 RAG의 핵심 요약 및 산업적 필요성

RAG는 "검색으로 근거를 찾고, 생성으로 답을 정리하는 구조"입니다.

기업이나 기관에서 RAG가 중요한 이유는 명확합니다.

- 내부 문서를 근거로 답할 수 있음
- 최신 자료를 반영하기 쉬움
- 답변의 출처를 설명하기 쉬움
- 일반 LLM보다 업무형 질의응답에 적합함
- 민감한 문서를 내부에 두고 활용하기 좋음

많은 조직은 민감한 문서를 외부 서비스로 보내기 어렵습니다.  
그래서 실무에서는 문서는 내부에 두고, 필요한 내용만 검색한 뒤, 로컬 또는 통제 가능한 모델과 함께 쓰는 형태의 RAG 구성이 자주 사용됩니다.
        

### 5.2 벡터 검색의 핵심 원리

앞 절에서 RAG가 왜 필요한지 봤다면, 이제는 **관련 문서를 어떻게 찾는지** 이해할 차례입니다.

이 절에서는 질문이 들어왔을 때 어떤 기준으로 관련 청크를 찾는지에 초점을 둡니다.

이 절의 초점은 RAG 정의를 다시 외우는 것이 아니라,  
**Retriever가 어떤 기준으로 문서를 찾는지 감을 잡는 것**입니다.


#### 5.2.1 Retriever는 무엇을 기준으로 문서를 찾을까?

질문과 문서가 정확히 같은 단어를 쓰지 않아도,  
의미가 비슷하면 찾아낼 수 있어야 검색 품질이 올라갑니다.

그래서 RAG에서는 보통 아래 두 가지를 함께 이해합니다.

1. **임베딩(Embedding)**: 문장을 숫자 벡터로 바꿔 의미를 비교할 수 있게 함
2. **벡터 DB(Vector DB)**: 그 벡터를 저장하고 비슷한 문서를 빠르게 찾음


#### 5.2.2 벡터 임베딩과 의미 기반 검색의 원리

이제 위 두 요소가 실제 검색에서 어떻게 연결되는지 보겠습니다.

핵심은 사용자의 질문과 문서를 **같은 의미 공간에 올려놓고**,
서로 가까운 후보를 찾는 것입니다.


##### 1) 전통적 검색(RDBMS)의 한계

전통적 검색은 **정확히 같은 단어**를 찾는 데는 강하지만,
표현이 조금만 달라져도 관련 문서를 놓치기 쉽습니다.

- "강아지"와 "개"
- "휴대폰 거래"와 "모바일폰 사용 시 유의사항"

처럼 의미는 비슷하지만 단어가 다르면 검색 성능이 떨어질 수 있습니다.
RAG에서는 이런 한계를 줄이기 위해 의미 기반 검색을 함께 사용합니다.
        

##### 2) 벡터 임베딩과 벡터 데이터베이스(Vector DB)

임베딩은 문장을 숫자 벡터로 바꾸는 과정입니다.

- "고양이가 물을 마신다"
- "강아지가 물을 마신다"

같은 문장은 단어가 완전히 같지 않아도 의미가 비슷하므로
벡터 공간에서도 가깝게 배치될 수 있습니다.

이렇게 바뀐 벡터를 저장하고 검색하는 저장소가 벡터 DB입니다.
        

<img src="image/vectorDB.webp" width="600">

**벡터 DB(Vector Database)는 무엇을 하는가?**

벡터 DB의 역할은 세 가지로 요약할 수 있습니다.

- 임베딩 벡터 저장
- 질문 벡터와 가까운 후보 검색
- 원문 텍스트와 메타데이터 함께 반환

즉, **임베딩은 의미를 숫자로 바꾸고**,
**벡터 DB는 그 숫자를 빠르게 찾게 해주는 저장소**입니다.
        

**임베딩과 벡터 DB의 관계 (핵심 정리)**

- **Embedding**: 문서를 검색 가능한 숫자 표현으로 변환
- **Vector DB**: 변환된 숫자를 저장하고 유사한 문서를 찾아줌

둘은 항상 같이 이해하면 됩니다.
RAG에서는 이 조합이 Retriever의 기본 토대가 됩니다.
        

##### 3) 자주 쓰는 벡터 DB와 이번 실습 선택

벡터 DB는 종류가 많지만, 입문 단계에서는 아래 정도만 가볍게 구분해도 충분합니다.

| 도구 | 한 줄 특징 | 장점 | 아쉬운 점 |
|---|---|---|---|
| **FAISS** | 빠른 유사도 검색 라이브러리 | 빠르고 가벼움, 로컬 실습에 적합, CPU/GPU 활용 가능 | 서비스형 DB처럼 운영 기능이 풍부한 편은 아님 |
| **Chroma** | 입문용 로컬 벡터 저장소 | 사용법이 단순하고 LangChain 예제가 많음 | 대규모 운영보다는 로컬/프로토타입에 더 잘 맞음 |
| **Pinecone** | 관리형 클라우드 벡터 DB | 서버 운영 부담이 적고 바로 붙이기 쉬움 | 외부 서비스 의존과 비용 고려가 필요함 |
| **Milvus** | 확장성 중심 오픈소스 벡터 DB | 대규모 데이터와 분산 환경에 강함 | 로컬 실습 기준으로는 구성과 운영이 더 무거움 |

이번 교안에서는 **FAISS**를 사용합니다.
이유는 로컬에서 바로 실행할 수 있고, 저장/검색/불러오기 흐름을 가장 단순하게 보여주기 좋기 때문입니다.
        

##### 4) 벡터 검색 전체 과정

벡터 검색은 아래 5단계로 이해하면 됩니다.
큰 흐름으로 보면 **1번은 Indexing**, **2~4번은 Retrieval**, **5번은 Generation**에 해당합니다.

1. 문서를 임베딩한다.
2. 질문도 임베딩한다.
3. 가까운 벡터를 찾는다.
4. 연결된 원문을 가져온다.
5. LLM이 그 문서를 바탕으로 답변한다.
        

**1. 문서 임베딩 생성 (Indexing)**
문서를 미리 숫자 벡터로 바꿔 저장합니다.
이 단계는 보통 인덱싱 시 한 번 수행합니다.
        

**2. 쿼리 임베딩 생성 (Retrieval 입력 준비)**
사용자 질문도 같은 방식으로 벡터로 변환합니다.
문서와 질문이 같은 공간에 있어야 유사도 비교가 가능합니다.
        

**3. k-NN 검색(k-Nearest Neighbors) (Retrieval)**
질문 벡터와 가장 가까운 문서 벡터 K개를 찾습니다.
이 단계가 실제 Retrieval의 핵심입니다.
        

**4. 유사 문서 반환 (Retrieval 결과)**
가까운 벡터에 연결된 원문 텍스트와 메타데이터를 반환합니다.
이제 LLM이 참고할 근거 문서가 준비됩니다.
        

**5. LLM이 반환 문서를 참고하여 답변 생성 (Generation)**
검색된 문서를 프롬프트에 넣어 전달하면,
LLM은 그 근거를 바탕으로 더 정확하고 신뢰도 높은 답변을 생성할 수 있습니다.
        

##### 5) 왜 벡터 검색이 중요한가?

- 환각(Hallucination) 감소  
- 희소·전문 정보 보완  
- 최신 정보 반영  
- 오픈소스 LLM의 부족한 사전지식 보완  
- 문서 기반 답변 → 신뢰도 향상

➡ 요약: 의미 기반 검색은 RAG 성능을 결정하는 핵심 요소이며
Retriever의 품질이 곧 RAG 전체 품질을 좌우한다.


##### 6) 작은 예제로 보는 벡터 검색


In [ ]:
from langchain_community.vectorstores import FAISS  # 임베딩 벡터를 저장하고 유사 문서를 찾는 벡터 DB입니다.
from langchain_openai import OpenAIEmbeddings  # 텍스트를 의미 기반 숫자 벡터로 바꿉니다.
from dotenv import load_dotenv  # .env 파일의 환경 변수를 현재 세션으로 불러옵니다.
load_dotenv(override=True)  # .env에 저장한 API 키를 현재 세션에 반영합니다.

# 예제 문장을 작은 지식 베이스처럼 사용합니다.
docs = [
    "고양이가 물을 마신다",
    "강아지가 물을 마신다",
    "사람이 커피를 마신다",
    "눈에 보이지 않는 것의 엄청난 힘",
    "메리 크리스마스"
]

# 각 문장에 함께 저장할 메타데이터도 같은 순서로 준비합니다.
metadatas = [
    {"id": 1, "topic": "동물"},
    {"id": 2, "topic": "동물"},
    {"id": 3, "topic": "사람"},
    {"id": 4, "topic": "문구"},
    {"id": 5, "topic": "인사"},
]

# 문장을 의미 기반 숫자 벡터로 바꿀 임베딩 모델을 준비합니다.
emb = OpenAIEmbeddings(model='text-embedding-3-small')

# 문장과 메타데이터를 함께 임베딩해 FAISS 벡터 DB로 만듭니다.
db = FAISS.from_texts(docs, emb, metadatas=metadatas)

# 질문도 같은 임베딩 공간에 올린 뒤 의미가 가까운 문장 3개를 찾습니다.
query = "고양이가 음료를 마신다"
results = db.similarity_search(query, k=3)

# 검색된 원문을 확인해 의미 기반 검색이 어떻게 동작하는지 봅니다.
for r in results:
    print(r.page_content)

In [ ]:
# 검색 결과와 함께 거리(score)도 확인합니다.
# FAISS의 기본 score는 거리이므로 값이 작을수록 더 가깝습니다.
scored_results = db.similarity_search_with_score(query, k=3)

for doc, score in scored_results:
    print(f"문장: {doc.page_content}")
    print(f"거리(score): {score:.4f}")
    print("-" * 40)

In [ ]:
# 벡터DB 내부 구조 확인
index = db.index

# 총 몇 개의 벡터가 저장되어 있는지
print("저장된 벡터 개수:", index.ntotal)

In [ ]:
# 벡터의 차원 수(embedding size)
print("벡터 차원:", index.d)

In [ ]:
# 첫 번째 벡터 가져오기 (0번째 위치)
vector = index.reconstruct(0)
print("첫 번째 벡터 일부:", vector[:10])  # 앞 10개만 출력

In [ ]:
# LangChain의 FAISS는 docstore 안에 원문을 저장함 
# 각 ID는 벡터와 원문 문서를 연결해, 검색 결과가 나오면 해당 원문을 다시 찾을 때 사용됩니다.
print("문서 ID 리스트:", db.docstore._dict.keys())

In [ ]:
# 가장 첫 번째 문서 ID 가져오기
first_id = list(db.docstore._dict.keys())[0]
# 해당 ID의 문서 객체 확인
print("첫 문서 내용:", db.docstore._dict[first_id].page_content)

In [ ]:
# docstore에 저장된 원문과 메타데이터를 한 번에 훑어봅니다.
print("=== 저장된 모든 문서와 메타데이터 ===")
for doc in db.docstore._dict.values():
    print("문서 내용:", doc.page_content)
    print("메타데이터:", doc.metadata)
    print("---")


➡ 의미적 유사성 기반 검색이 정상적으로 작동

##### 저장 후 다시 불러오기

작은 예제 DB를 저장했다가 다시 불러오겠습니다.  
이렇게 하면 벡터 DB를 매번 새로 만들지 않고 다시 사용할 수 있습니다.


In [ ]:
DB_PATH = "./my_faiss_index_test"  # 로컬에 저장할 벡터 DB 폴더 경로입니다.
db.save_local(DB_PATH)  # 현재 메모리의 FAISS 인덱스를 디스크에 저장합니다.

print(f"'{DB_PATH}' 경로에 저장 완료되었습니다.")

In [ ]:
# 로컬에서 불러오기
# 주의: 저장할 때 썼던 임베딩 모델(emb)과 동일한 객체를 넘겨줘야 합니다.
new_db = FAISS.load_local(
    folder_path=DB_PATH, 
    embeddings=emb, 
    allow_dangerous_deserialization=True  # 최신 버전 필수 옵션
)

print("DB 로드 완료!")

# 4. 잘 불러와졌는지 테스트
query = "고양이는 무엇을 마시나요?"
docs = new_db.similarity_search(query)
print(docs[0].page_content)

##### 참고

기존 벡터 DB에 새 정보를 추가할 때, 문자열 목록은 `add_texts()`로 넣고  
PDF를 청킹해 만든 `Document` 리스트는 `add_documents()`로 추가할 수 있습니다.

PDF를 불러오고 청킹하는 흐름은 바로 뒤에서 다시 살펴봅니다.



## 6. RAG 파이프라인 구성 실습

이번 실습에서는 **`금융투자협회_투자길라잡이_2018.pdf`** 파일을 사용합니다.

이 문서는 계좌 개설, 계좌 관리, 모바일 거래, 분쟁 사례 등 투자자가 실제로 참고할 만한 안내 정보를 담고 있습니다.  
그래서 **일반 LLM이 답하는 경우**와 **문서를 근거로 답하는 RAG**를 비교해 보기 좋습니다.

단계 기준으로 보면, 이 실습은 **Indexing -> Retrieval -> Generation**이 어떻게 이어지는지 한 번에 연결해 보는 기본형 RAG 실습입니다.

실습은 아래 순서로 진행됩니다.

1. PDF를 불러오고 청킹한다
2. 임베딩 후 벡터 DB(FAISS)에 저장한다
3. LCEL 문법으로 RAG 체인을 구성한다
4. 같은 질문을 일반 LLM과 RAG 체인에 각각 넣어 비교한다


### 6.1 비교 질문

이번 비교의 초점은 **일반 상식만으로는 답하기 어려운, 문서를 봐야 알 수 있는 구체적 정보**를 RAG가 어떻게 보완하는지 보는 것입니다.

핵심은 아래 두 가지입니다.
- 일반 LLM은 금융 일반론은 말할 수 있어도, 이런 전문적이거나 폐쇄적인 정보는 놓치기 쉽다
- RAG는 문서를 근거로 제도 이름, 절차, 유의사항 같은 구체적 정보를 찾아 답할 수 있다

> **Q. "피싱사기로 개인정보를 알려줬을 때 가까운 은행에 등록 요청하라고 한 시스템 이름은 무엇인가요?"**


In [ ]:
# 필요한 패키지 임포트
from langchain_community.document_loaders import PyPDFLoader  # PDF를 페이지 단위 문서로 읽어옵니다.
from langchain_text_splitters import RecursiveCharacterTextSplitter  # 문맥을 최대한 살리며 긴 문서를 여러 청크로 나눕니다.

from langchain_openai import OpenAIEmbeddings, ChatOpenAI  # OpenAI 임베딩 모델과 채팅 모델을 함께 불러옵니다.
from langchain_community.vectorstores import FAISS  # 임베딩 벡터를 저장하고 유사 문서를 찾는 벡터 DB입니다.

from langchain_core.prompts import ChatPromptTemplate  # 여러 메시지 역할을 묶는 채팅 프롬프트 템플릿입니다.
from langchain_core.runnables import RunnablePassthrough, RunnableLambda  # 입력값을 그대로 넘기거나, 파이썬 함수를 체인 단계로 감쌀 때 씁니다.
from langchain_core.output_parsers import StrOutputParser  # 모델 응답 객체에서 텍스트만 꺼내는 파서입니다.

import os  # 파일 경로와 환경 변수를 다룰 때 쓰는 표준 라이브러리입니다.
from dotenv import load_dotenv  # .env 파일의 환경 변수를 현재 세션으로 불러옵니다.
from pdf_utils import clean_pdf_documents  # PDF 정제 공통 유틸을 불러옵니다.
load_dotenv(override=True) # 0. 환경 변수 로드 (OPENAI_API_KEY는 .env에 저장했다고 가정)

In [ ]:
# 1. 문서 로드 & 청크 분할
PDF_PATH = "data/금융투자협회_투자길라잡이_2018.pdf"
# 정제한 문서를 저장할 벡터 DB 경로입니다.
DB_PATH = "./faiss_index_kofia_guide_cleaned"

# 임베딩 모델은 문서와 질문을 같은 벡터 공간으로 바꿀 때 사용합니다.
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 이미 만들어 둔 벡터 DB가 있으면 다시 만들지 않고 바로 재사용합니다.
if os.path.exists(DB_PATH):
    print(f"기존 벡터 DB를 '{DB_PATH}'에서 불러옵니다...")
    # 저장해 둔 FAISS 벡터 DB를 다시 불러옵니다.
    vectorstore = FAISS.load_local(
        folder_path=DB_PATH,
        embeddings=embeddings,
        allow_dangerous_deserialization=True
    )
    print("DB 로드 완료!")

else:
    print("저장된 DB가 없습니다. 문서를 로드하고 새로 생성합니다...")

    # PDF를 페이지 단위 문서로 불러옵니다.
    loader = PyPDFLoader(PDF_PATH)
    docs = loader.load()
    # clean_pdf_documents()는 머리말/쪽번호/깨진 글리프를 먼저 정리해 청킹 품질을 맞춥니다.
    docs = clean_pdf_documents(docs)

    # 문단 -> 문장 -> 단어 순서로 경계를 시도합니다.
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=100,
        separators=["\n\n", ".", " ", ""],
    )

    split_docs = text_splitter.split_documents(docs)

    # 각 청크에 간단한 ID를 붙여 두면 나중에 추적하기 쉽습니다.
    for idx, d in enumerate(split_docs):
        d.metadata["id"] = idx

    print(f"원본 페이지 수: {len(docs)}")
    print(f"분할된 청크 수: {len(split_docs)}")

    # 문서 청크와 임베딩으로 FAISS 벡터 DB를 만듭니다.
    vectorstore = FAISS.from_documents(split_docs, embeddings)
    vectorstore.save_local(DB_PATH)
    print(f"벡터 DB가 '{DB_PATH}' 경로에 저장되었습니다.")

# retriever는 질문을 받아 관련 청크를 찾아주는 검색기입니다.
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

In [ ]:
# 3. 답변을 생성할 채팅 모델을 준비합니다.
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.1,       # 최대한 사실 위주의 응답을 위해 낮게 설정
)

In [ ]:
# 4. 검색된 문서 리스트를 프롬프트에 넣기 좋은 하나의 문자열로 바꿉니다.
def format_docs(docs):  # 검색 문서를 프롬프트에 넣을 하나의 문자열로 합칩니다.
    """검색된 Document 리스트를 하나의 문자열로 합치는 함수."""
    parts = []  # 문서별 문자열 조각을 순서대로 모읍니다.
    for d in docs:
        # 페이지 정보가 있으면 함께 붙여서 나중에 근거를 확인하기 쉽게 합니다.
        page = d.metadata.get("page", "N/A")
        parts.append(f"[page {page}]\n{d.page_content}")  # 페이지 번호와 본문을 함께 붙입니다.
    return "\n\n".join(parts)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate  # 여러 메시지 역할을 묶는 채팅 프롬프트 템플릿입니다.

# 비교를 위해 두 프롬프트의 역할과 답변 형식은 최대한 같게 맞춥니다.
base_system_instruction = (
    "너는 금융 질문에 답하는 AI 어시스턴트야. "
    "답은 가능하면 핵심 명사구나 짧은 한 줄로만 하고, 불릿이나 긴 설명은 하지 말 것."
)

# RAG 체인은 문서 컨텍스트만 근거로 답하도록 제한합니다.
rag_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            base_system_instruction
            + " 이번에는 반드시 제공된 문서 컨텍스트 안의 내용만 근거로 답해. "
            + "컨텍스트에 정답이 없으면 '제공된 문서 내용에서는 찾을 수 없습니다'라고 답해.",
        ),
        (
            "user",
            (
                "질문: {question}\n\n"
                "참고할 문서 컨텍스트:\n"
                "--------------------\n"
                "{context}\n"
                "--------------------\n"
                "위 문서 내용만 근거로 답해줘."
            ),
        ),
    ]
)

# 비교용 체인은 같은 형식을 유지하되, 문서 컨텍스트 없이 일반 지식만 사용합니다.
no_rag_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            base_system_instruction
            + " 이번에는 참고할 문서 컨텍스트가 없다고 가정하고, 네 일반 지식만으로 답해. "
            + "모르면 억지로 지어내지 말고 잘 모르겠다고 답해.",
        ),
        (
            "user",
            (
                "질문: {question}\n\n"
                "참고할 문서 컨텍스트는 제공되지 않습니다.\n"
                "네 일반 지식만으로 답해줘."
            ),
        ),
    ]
)


In [ ]:
# 6. LCEL로 RAG 체인
rag_chain = (  # 질문 -> 검색 -> 문맥 조립 -> 답변 생성 흐름입니다.
    {
        "question": RunnablePassthrough(),  # 질문 그대로 전달
        "context": retriever | RunnableLambda(format_docs),  # 검색 결과를 문맥 문자열로 변환
    }
    | rag_prompt  # question과 context를 답변용 프롬프트로 묶습니다.
    | llm  # 문맥을 바탕으로 답을 생성합니다.
    | StrOutputParser()  # 모델 응답에서 텍스트만 꺼냅니다.
)

# 검색 없이 질문만 넣는 비교용 체인입니다.
no_rag_chain = no_rag_prompt | llm | StrOutputParser()

# 실제 비교에 사용할 질문입니다.
question = "피싱사기로 개인정보를 알려줬을 때 가까운 은행에 등록 요청하라고 한 시스템 이름은 무엇인가요?"

print("=== [1] RAG 체인 결과 ===")
rag_answer = rag_chain.invoke(question)  # 질문 하나를 넣어 RAG 답변을 만듭니다.
print(rag_answer)

print("\n\n=== [2] RAG 없이 LLM만 사용한 결과 ===")
no_rag_answer = no_rag_chain.invoke({"question": question})  # 같은 질문을 일반 지식만으로 답하게 합니다.
print(no_rag_answer)

핵심은 **일반 LLM은 문서 고유 표현을 틀리기 쉽고, RAG는 검색된 문서 근거를 바탕으로 더 정확히 답할 수 있다**는 점입니다.



## 7. 기본형 RAG의 한계 (Naive RAG)

실무에서는 아주 많은 문서와 청크가 한꺼번에 검색 대상이 되고, 문서 형식과 표현 방식도 제각각입니다. 이런 환경에서는 가장 기본형 RAG만으로는 필요한 근거를 안정적으로 찾기 어려워집니다.

그 결과 정답이 들어 있는 문서를 놓쳐서 잘못된 정보를 말하거나, 애매한 근거만 잡혀 답변이 흐려지거나, 필요한 근거를 찾지 못해 제대로 답하지 못하는 일이 생깁니다.

중요한 점은 성능이 안 좋을 때 이를 막연히 "RAG가 안 된다"고 보지 않고, 어느 단계에서 흔들리는지 나누어 진단하는 것입니다.

그래야 원인에 맞는 개선 방향도 잡을 수 있습니다. 

### 7.1 단계별로 진단하는 기준

여기서 **Naive RAG**는 문서를 검색해 프롬프트에 붙여 답하는 **가장 기본형 RAG** 정도로 이해하면 됩니다.

기본형 RAG는 보통 아래 세 단계로 나누어 볼 수 있습니다.

- **Indexing**: 문서를 불러오고, 정리하고, 청킹하고, 임베딩해 벡터 DB에 넣는 단계
- **Retrieval**: 질문과 가까운 청크를 찾는 단계
- **Generation**: 찾은 문맥을 바탕으로 최종 답변을 만드는 단계

문서마다 형식이 다르고 표현이 제각각일 때는, 이 세 단계를 따로 봐야 문제를 더 정확히 찾을 수 있습니다.  
예를 들어 문서 추출이나 청킹이 어색한지, 검색이 엉뚱한 청크를 가져오는지, 답변 단계에서 근거 없이 추측하는지를 나누어 봐야 개선 방향도 선명해집니다.

아래 표는 Indexing, Retrieval, Generation 단계에서 기본형 RAG가 자주 부딪히는 문제를 정리한 것입니다.  
모든 문제가 단계 자체의 필연적 한계는 아니고, 단순한 설계나 설정 때문에 생기는 경우도 함께 포함합니다.  


### 7.2 단계별 한계 정리

<table style="width:100%; border-collapse:collapse; font-size:14px;">
  <thead>
    <tr>
      <th style="border:1px solid #ccc; padding:8px; text-align:left;">단계</th>
      <th style="border:1px solid #ccc; padding:8px; text-align:left;">문제 지점</th>
      <th style="border:1px solid #ccc; padding:8px; text-align:left;">성격</th>
      <th style="border:1px solid #ccc; padding:8px; text-align:left;">왜 문제가 되나</th>
      <th style="border:1px solid #ccc; padding:8px; text-align:left;">대표 보완 방향</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td rowspan="3" style="border:1px solid #ccc; padding:8px; vertical-align:top;"><strong>Indexing</strong></td>
      <td style="border:1px solid #ccc; padding:8px;">PDF/스캔 텍스트 추출 오류</td>
      <td style="border:1px solid #ccc; padding:8px;">입력 품질 제약</td>
      <td style="border:1px solid #ccc; padding:8px;">문서 내용이 깨지면 이후 검색과 답변도 함께 흔들림</td>
      <td style="border:1px solid #ccc; padding:8px;">OCR, 전처리, 추출 결과 검수</td>
    </tr>
    <tr>
      <td style="border:1px solid #ccc; padding:8px;">문서 구조를 무시한 단순 청킹</td>
      <td style="border:1px solid #ccc; padding:8px;">단순한 처리 방식에서 생기는 문제</td>
      <td style="border:1px solid #ccc; padding:8px;">필요한 문맥이 잘리거나 다른 주제가 한 청크에 섞임</td>
      <td style="border:1px solid #ccc; padding:8px;">재귀적 청킹, 구조 기반 청킹</td>
    </tr>
    <tr>
      <td style="border:1px solid #ccc; padding:8px;">임베딩 모델의 표현력 부족</td>
      <td style="border:1px solid #ccc; padding:8px;">모델/도구 한계</td>
      <td style="border:1px solid #ccc; padding:8px;">비슷한 뜻을 약하게 잡아 관련 청크를 놓칠 수 있음</td>
      <td style="border:1px solid #ccc; padding:8px;">임베딩 평가, 도메인에 맞는 모델 선택</td>
    </tr>
    <tr>
      <td rowspan="3" style="border:1px solid #ccc; padding:8px; vertical-align:top;"><strong>Retrieval</strong></td>
      <td style="border:1px solid #ccc; padding:8px;">검색 방식 하나만 사용하는 구조</td>
      <td style="border:1px solid #ccc; padding:8px;">단순한 검색 설계에서 생기는 문제</td>
      <td style="border:1px solid #ccc; padding:8px;">dense만 쓰거나 keyword만 쓰면 한쪽 약점이 그대로 남음</td>
      <td style="border:1px solid #ccc; padding:8px;">Hybrid Retrieval, 검색 조합</td>
    </tr>
    <tr>
      <td style="border:1px solid #ccc; padding:8px;">질문이 짧거나 모호함</td>
      <td style="border:1px solid #ccc; padding:8px;">입력 표현 한계</td>
      <td style="border:1px solid #ccc; padding:8px;">검색기가 찾을 단서가 부족해 엉뚱한 문서를 가져올 수 있음</td>
      <td style="border:1px solid #ccc; padding:8px;">Query Rewriting, 질문 구체화</td>
    </tr>
    <tr>
      <td style="border:1px solid #ccc; padding:8px;">상위 검색 결과의 다양성 부족</td>
      <td style="border:1px solid #ccc; padding:8px;">랭킹/다양성 한계</td>
      <td style="border:1px solid #ccc; padding:8px;">비슷한 청크만 반복되면 다른 핵심 근거가 컨텍스트에 못 들어갈 수 있음</td>
      <td style="border:1px solid #ccc; padding:8px;">Reranking, 중복 제거, 메타데이터 필터링</td>
    </tr>
    <tr>
      <td rowspan="3" style="border:1px solid #ccc; padding:8px; vertical-align:top;"><strong>Generation</strong></td>
      <td style="border:1px solid #ccc; padding:8px;">근거가 약할 때 추측으로 답함</td>
      <td style="border:1px solid #ccc; padding:8px;">모델 고유 한계</td>
      <td style="border:1px solid #ccc; padding:8px;">문서에 없는 내용을 그럴듯하게 지어낼 수 있음</td>
      <td style="border:1px solid #ccc; padding:8px;">근거 중심 프롬프트, 답변 제약</td>
    </tr>
    <tr>
      <td style="border:1px solid #ccc; padding:8px;">문맥이 너무 많거나 중복됨</td>
      <td style="border:1px solid #ccc; padding:8px;">앞 단계 결과와 모델 사용 한계가 겹친 문제</td>
      <td style="border:1px solid #ccc; padding:8px;">핵심 근거가 묻혀 답이 장황해지거나 흐려질 수 있음</td>
      <td style="border:1px solid #ccc; padding:8px;">검색 결과 정리, Reranking, 컨텍스트 압축</td>
    </tr>
    <tr>
      <td style="border:1px solid #ccc; padding:8px;">검색된 근거보다 사전지식에 기대는 경우</td>
      <td style="border:1px solid #ccc; padding:8px;">모델 사용 특성</td>
      <td style="border:1px solid #ccc; padding:8px;">문서 근거와 어긋난 답을 할 수 있음</td>
      <td style="border:1px solid #ccc; padding:8px;">출처 제시, 답변 규칙 강화</td>
    </tr>
  </tbody>
</table>


### 7.3 왜 Advanced RAG가 필요한가?

여기서 **Advanced RAG**는 Naive RAG의 한계를 줄이기 위해, Indexing / Retrieval / Generation 단계에 여러 보완 기법을 추가한 RAG를 뜻합니다.  
예를 들어 청킹 전략, 질문 재작성, 하이브리드 검색, 재정렬, 후처리 같은 방법이 여기에 들어갑니다.

표에서 보듯 Naive RAG의 문제는 한 군데만 손봐서 해결되지 않습니다.
문서를 나누는 방식, 질문을 만드는 방식, 검색 결과를 정리하는 방식, 답변 규칙을 함께 조정해야 품질이 안정됩니다.

여기서는 그중에서도 앞단에서 바로 개선 효과를 보기 쉬운 **Chunking**과 **Query Rewriting**을 먼저 봅니다.


## 8. Advanced RAG 입문: Chunking과 Query Rewriting

앞에서 본 단계 기준으로 보면, 이 장은 **Indexing**과 **Pre-Retrieval**을 조금 더 정교하게 만드는 입문 구간입니다.
즉, 문서를 더 잘 저장하는 방법과 질문을 더 잘 찾히게 바꾸는 방법을 먼저 봅니다.

| 먼저 볼 질문 | 연결되는 개선 포인트 |
|---|---|
| 문서가 왜 어색하게 잘려 필요한 근거를 놓칠까? | 고급 Chunking |
| 질문을 왜 한 번 더 다듬어야 검색이 잘 될까? | Query Rewriting |


### 8.1 고급 Chunking 기법

Chunking은 **Indexing 단계**에서 문서를 어떤 단위로 저장할지 정하는 작업입니다.  
그래서 검색 품질에 직접 영향을 주는 요소입니다.  
같은 임베딩 모델과 벡터DB를 써도, 문서를 어떻게 나누느냐에 따라 검색 결과가 달라질 수 있습니다.

문서를 어떤 단위로 나눠 저장하느냐는 이후 검색 품질에 영향을 줍니다.

이 절에서는 다음 세 가지를 다룹니다.

1. 고정 길이 청크 (Fixed-size Chunking)  
2. 재귀적 청크 (Recursive Text Splitting)  
3. 구조 기반 청크 (Structure-aware Chunking)  


#### 8.1.1 왜 Chunking이 중요한가?

임베딩 모델과 벡터DB는 **청크 단위로 검색**을 한다.

- 벡터DB에 저장되는 기본 단위: **청크(chunk)**  
- 검색은 문서 전체가 아니라 청크 벡터들을 대상으로 수행  
- 따라서,
  - 청크가 너무 길면 → 하나의 청크 안에 여러 주제가 섞여 **노이즈 증가**  
  - 청크가 너무 짧으면 → 의미 단위가 깨져서 **문맥 정보 손실**

**좋은 청크의 조건**

1. **완결된 의미 단위**여야 한다. (문단·섹션 단위 등)  
2. **너무 길지도, 너무 짧지도 않아야** 한다. (보통 200~500 tokens 권장)  
3. 서로 **적당히 겹침(overlap)** 이 있어 컨텍스트를 안정적으로 이어줌

이제 각각의 대표적인 청크 방식과 실습 코드를 보겠습니다.

#### 8.1.2 고정 길이 청크 (Fixed-size Chunking)

가장 단순하고 이해하기 쉬운 방식입니다.

- 일정 길이(문자 수 또는 토큰 수) 기준으로 **앞에서부터 잘라 나가는 방식**
- 장점
  - 구현이 매우 쉬움
  - 데이터 형식과 상관없이 동일한 전략 적용 가능
- 단점
  - 문장이 중간에서 끊길 수 있음
  - 한 청크 = 한 의미가 잘 안 맞음
  - 문단/제목/섹션 등 문서 구조를 전혀 고려하지 않음

실제 출력 차이의 크기는 문서 상태에 따라 달라질 수 있습니다.

> **고정 길이 청크는 가장 단순한 출발점으로 보고,  
> 필요하면 재귀적 청크나 구조 기반 청크와 함께 비교하는 편이 좋습니다.**

##### ① 실습: PDF 로드 + 전체 텍스트 합치기

아래 코드는 **PDF를 불러오고, 잡음을 조금 정리한 뒤 실습에 쓸 텍스트를 준비하는 코드**입니다.  
세부 함수 구현을 모두 이해할 필요는 없고, 여기서는 **`docs`를 만들고 `full_text`를 준비한다**는 흐름만 보면 충분합니다.


In [ ]:
# 아래 코드는 PDF를 읽고, 실습에 사용할 본문 예시를 준비하는 최소 코드입니다.
from langchain_community.document_loaders import PyPDFLoader  # PDF를 페이지 단위 문서로 읽어옵니다.

PDF_PATH = "data/금융투자협회_투자길라잡이_2018.pdf"

loader = PyPDFLoader(PDF_PATH)
# clean_pdf_documents()는 PDF 추출 과정에서 섞인 잡음을 먼저 정리합니다.
docs = clean_pdf_documents(loader.load())

print("페이지 수:", len(docs))
print("예시로 볼 본문 페이지:", docs[58].metadata.get("page", 0) + 1)
print("본문 미리보기:")
print(docs[58].page_content[:320])

- `docs`는 페이지별 `Document` 리스트입니다.
- 아래 청킹 예시는 `full_text` 전체를 같은 입력으로 사용합니다.
- 첫 청크는 비슷할 수 있어, 여기서는 둘째 청크를 봅니다.


In [ ]:
# split_text()는 하나의 긴 문자열을 입력으로 받으므로 페이지 텍스트를 먼저 이어 붙입니다.
full_text = "\n".join(doc.page_content for doc in docs)

print("전체 텍스트 길이(문자 수):", len(full_text))
# 이제 이 긴 문자열을 여러 방식으로 나눠 비교합니다.

##### ② 실습: CharacterTextSplitter로 고정 길이 청크

`CharacterTextSplitter`는 정한 기준으로 텍스트를 잘라 고정 길이 청크를 만드는 도구입니다.


In [ ]:
from langchain_text_splitters import CharacterTextSplitter  # 고정 기준으로 문서를 잘라보는 분할기입니다.

# CharacterTextSplitter는 길이 기준으로 비교적 단순하게 텍스트를 자릅니다.
fixed_splitter = CharacterTextSplitter(
    separator="",        # 의미 경계를 따로 보지 않고 글자 수 기준으로 자릅니다.
    chunk_size=500,       # 한 청크의 최대 길이입니다. 여기서는 문자 수 기준입니다.
    chunk_overlap=100,    # 앞 청크 끝부분 100자를 다음 청크에도 겹쳐 넣습니다.
    length_function=len,  # 길이를 잴 때 문자 수(len)를 사용합니다.
)

# 전체 문서를 잘라 실제 청크 개수를 확인합니다.
fixed_chunks = fixed_splitter.split_text(full_text)

print("고정 길이 청크 개수:", len(fixed_chunks))
print("고정 길이 청크 예시(둘째 청크):")
print(fixed_chunks[1][:300].replace("\n", " "))

- `chunk_size`와 `chunk_overlap`은 나중에 성능 튜닝의 핵심 하이퍼파라미터가 됩니다.
- 여기서는 의미 경계보다 **글자 수 기준**으로 먼저 자르는 가장 단순한 방식을 보여줍니다.
- 이 방식은 정확하진 않지만 아무 문서에나 바로 적용 가능하다는 장점이 있습니다.


#### 8.1.3 재귀적 청크 (Recursive Text Splitting)

`RecursiveCharacterTextSplitter`는 큰 경계부터 먼저 쓰고, 너무 길면 더 작은 경계로 내려가며 자릅니다.

보통 **문단 -> 문장 비슷한 경계 -> 단어 -> 문자** 순서로 나눈다고 이해하면 충분합니다.
이번 PDF는 줄바꿈 잡음이 많아서, 여기서는 `"\n"` 경계는 쓰지 않고 더 자연스러운 경계를 먼저 사용합니다.


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter  # 문맥을 최대한 살리며 긴 문서를 여러 청크로 나눕니다.

# 이번 PDF는 줄바꿈이 의미 경계보다 추출 잡음에 가까워, 줄 단위 분리는 제외합니다.
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,       # 한 청크의 최대 길이입니다.
    chunk_overlap=100,    # 청크 사이에 100자를 겹쳐 문맥 단절을 줄입니다.
    separators=[
        "\n\n",  # 문단 경계를 가장 먼저 시도합니다.
        ".",       # 그다음 문장 비슷한 경계를 시도합니다.
        " ",        # 필요하면 단어 단위까지 내려갑니다.
        "",         # 마지막에는 문자 단위로 강제 분할합니다.
    ],
)

# 전체 문서를 잘라 실제 청크 개수를 확인합니다.
recursive_chunks = recursive_splitter.split_text(full_text)

print("재귀적 청크 개수:", len(recursive_chunks))
print("재귀적 청크 예시(둘째 청크):")
print(recursive_chunks[1][:300].replace("\n", " "))


두 결과가 비슷하게 보이거나, 재귀 청크도 어색하게 시작하는 것처럼 보일 수 있는 이유는 **PDF 추출 결과 자체에 잡음이 많기 때문**입니다.

특히 이 실습 PDF는 줄이 중간에 끊겨 있어, 줄바꿈을 그대로 경계로 쓰면 오히려 부자연스럽게 잘릴 수 있습니다.
그래서 위 예시에서는 `"\n"` 경계를 제외하고, 문단과 문장 비슷한 경계를 먼저 사용했습니다.

그래도 PDF 원문 자체가 고르지 않으면 차이가 아주 크게 보이지 않을 수 있습니다.
핵심은 특정 문장이 어떻게 나왔는지보다, 경계를 잡는 방식을 어떻게 바꾸는지 이해하는 것입니다.

이럴 때는 아래처럼 **문서 구조를 먼저 나눈 뒤** 다시 청킹하면 차이가 더 잘 드러납니다.


#### 8.1.4 문서 구조 기반 청크 (Structure-aware Chunking)

문서에 사례 제목, 조문 번호, 절 제목처럼 **반복되는 형식**이 있으면 그 형식을 먼저 경계로 쓸 수 있습니다.  
특히 법전, 규정집, 약관처럼 `제1조`, `제2조`, `제3조`처럼 형식이 일정한 문서는 구조 기반 청킹이 더 잘 맞습니다.  
여기서는 `금융투자상품 불완전판매 사례 1/2/3`처럼 반복되는 제목을 기준으로 먼저 나누는 예시만 간단히 봅니다.  

핵심은 `사례 1` 내용과 `사례 2` 내용이 같은 청크에 섞이지 않게, 먼저 구조 경계를 정하는 것입니다.  

<img src="image/case_demo-088.png" width="520">

<img src="image/case_demo-089.png" width="520">


##### ① 실습: 사례 제목 기준으로 먼저 나누기


In [ ]:
import re  # 정규 표현식으로 패턴을 찾을 때 쓰는 표준 라이브러리입니다.

# 사례 1/2/3이 이어지는 본문 일부를 예시로 사용합니다.
structure_demo_docs = docs[87:90]  # 사례 1~3이 보이는 페이지 구간만 잘라 씁니다.
structure_demo_text = "\n\n".join(doc.page_content for doc in structure_demo_docs)  # 여러 페이지를 하나의 문자열로 이어 붙입니다.

# 반복되는 사례 제목을 기준으로 먼저 나눕니다.
case_pattern = re.compile(r"금융투자상품\s*불완전판매\s*사례\s*\d+")
case_matches = list(case_pattern.finditer(structure_demo_text))  # 사례 제목이 시작하는 위치를 모두 찾습니다.

structured_sections = []  # 사례별로 잘린 본문을 담습니다.
for i, match in enumerate(case_matches):
    start = match.start()  # 현재 사례가 시작하는 위치입니다.
    end = case_matches[i + 1].start() if i + 1 < len(case_matches) else len(structure_demo_text)  # 다음 사례 시작 전까지를 현재 사례 범위로 봅니다.
    section = structure_demo_text[start:end].strip()
    if section:
        structured_sections.append(section)

# 제목이 안 잡히면 각 페이지 본문을 그대로 예시로 사용합니다.
if not structured_sections:
    structured_sections = [doc.page_content.strip() for doc in structure_demo_docs if doc.page_content.strip()]

case_titles = []  # 화면에 보여 줄 사례 제목 목록입니다.
for section in structured_sections:
    match = case_pattern.search(section)
    case_titles.append(match.group(0) if match else section.split("\n", 1)[0])

sample_section = structured_sections[0] if structured_sections else ""  # 대표 예시 하나를 바로 꺼내 쓸 수 있게 저장합니다.


In [ ]:
print("구조 데모에 사용한 페이지: 88~90")
print(f"총 사례 개수: {len(structured_sections)}")
print("정규표현식:", case_pattern.pattern)
print("\n=== 사례 제목 목록 ===")
for title in case_titles:  # 사례 제목이 예상대로 분리됐는지 확인합니다.
    print("-", title)

for idx, section in enumerate(structured_sections[:2], 1):  # 각 사례 앞부분만 보고 분리 결과를 빠르게 점검합니다.
    print(f"\n=== 사례 {idx} 앞부분 ===")
    print("\n".join(section.splitlines()[:6]))

##### ② 정리

이 예시에서는 페이지 단위와 결과가 크게 다르지 않을 수 있습니다.
그래도 구조 기반 청킹의 핵심은 `사례 1`, `사례 2`, `사례 3`처럼 **서로 다른 사례가 한 청크에 섞이지 않게 먼저 경계를 정한다**는 것입니다.
사례 하나가 너무 길다면, 이렇게 나눈 뒤 그 안을 다시 더 작은 검색용 청크로 자를 수 있습니다.


### 8.2 Pre-Retrieval 최적화: Query Transformation

지금까지는 문서를 잘 쪼개는 과정(Indexing)에 집중했습니다.  
이제는 검색 품질을 좌우하는 또 다른 요소인 **질문(Query)** 쪽을 볼 차례입니다.

아래 비교에서는 앞의 RAG 실습에서 만든 `retriever`를 그대로 사용합니다.

RAG 파이프라인에서 검색 직전 질문을 다듬는 단계를 **Pre-Retrieval**이라고 합니다.
핵심은 사용자의 짧고 모호한 질문을 **문서가 잘 찾을 수 있는 표현**으로 바꾸는 것입니다.


#### 8.2.1 Query Rewriting (LLM 기반)

Query Rewriting은 사용자 질문을 문서 검색에 더 잘 맞는 표현으로 바꾸는 방법입니다.

핵심은 하나입니다. **사용자 질문의 뜻은 유지하되, 문서에 실제로 쓰인 표현으로 바꾸는 것**입니다.

예를 들어 `휴대폰으로 주식 거래할 때 뭐 조심해야 하지?`는 문서 검색에서는 `모바일폰 사용 시 유의사항`, `금융거래`, `비밀번호` 같은 표현이 더 잘 맞을 수 있습니다.


##### 실습 1: 구어체 질문을 검색 표현으로 바꾸기

먼저 사용자의 짧은 질문을 검색에 더 잘 맞는 표현으로 바꿔보겠습니다.


In [ ]:
from langchain_core.prompts import PromptTemplate  # 프롬프트 텍스트 틀을 만드는 도구입니다.
from langchain_openai import ChatOpenAI  # 질문 재작성에 사용할 채팅 모델입니다.

# temperature=0은 재작성 결과를 최대한 일정하게 유지하려는 설정입니다. gpt-5-nano 대신 gpt-4o-mini를 써도 됩니다.
llm = ChatOpenAI(model="gpt-5-nano", temperature=0)

# prompt와 query를 받아 재작성된 질문 문자열만 돌려주는 보조 함수입니다.
def rewrite_query(prompt, query):  # 질문 재작성 결과 텍스트만 돌려주는 보조 함수입니다.
    return llm.invoke(prompt.format(query=query)).content  # 응답 객체에서 텍스트만 꺼냅니다.

# from_template()는 {query} 자리에 실제 질문을 넣어 프롬프트를 완성합니다.
basic_rewrite_prompt = PromptTemplate.from_template("""
당신은 금융투자협회의 투자자 안내서를 검색하는 보조 AI입니다.
사용자의 질문을 문서 검색에 더 잘 맞는 문장으로 다시 쓰세요.

규칙:
- 핵심 의도는 유지할 것
- 구어체는 더 또렷한 검색 문장으로 바꿀 것
- 필요하면 문서의 공식 용어를 보강할 것
- 없는 사실은 추가하지 말 것

질문: {query}
검색용 쿼리:
""")

query = "휴대폰으로 주식 거래할 때 뭐 조심해야 하지?"  # 원래 사용자의 질문입니다.
new_query = rewrite_query(basic_rewrite_prompt, query)  # 검색에 더 잘 맞는 표현으로 다시 씁니다.

print(f"Original: {query}")
print(f"Rewritten: {new_query}")


##### 실습 2: 공식 용어 보강하기

이번에는 질문 의도는 유지한 채, 문서의 공식 용어를 더 적극적으로 붙여보겠습니다.


In [ ]:
# 이번 프롬프트는 문서의 공식 용어를 더 적극적으로 반영하도록 유도합니다.
domain_rewrite_prompt = PromptTemplate.from_template("""
당신은 금융투자협회의 투자자 안내서를 검색하려는 사용자를 돕는 AI입니다.
사용자의 질문을 벡터 DB 검색에 더 잘 맞는 문장으로 다시 쓰세요.

규칙:
- 질문 의도는 유지할 것
- 문서의 공식 용어를 반영할 것
- 없는 사실은 추가하지 말 것

[변환 예시]
- 휴대폰, 앱, 스마트폰 거래 -> 모바일폰 사용 시 유의사항, 금융거래
- 계좌를 맡김, 대신 운용 -> 투자일임, 일임매매, 계좌관리 위임
- 멋대로 사고팖 -> 임의매매
- 무리한 투자 권유 -> 부당권유, 불완전판매

질문: {query}
검색용 쿼리:
""")

query = "직원한테 계좌 맡겨도 되는 거야?"  # 더 모호한 질문을 예시로 넣습니다.
new_query = rewrite_query(domain_rewrite_prompt, query)  # 공식 용어가 반영되었는지 확인합니다.

print(f"Original: {query}")
print(f"Rewritten: {new_query}")

In [ ]:
# 검색 결과를 페이지 번호와 본문 일부만 보이게 정리합니다.
def print_search_preview(title, results, max_docs=3):  # 검색 결과 상위 문서를 짧게 미리 봅니다.
    print(title)
    for i, doc in enumerate(results[:max_docs], 1):
        page = doc.metadata.get("page", 0) + 1  # 0부터 시작하는 페이지 번호를 사람이 보는 번호로 바꿉니다.
        snippet = doc.page_content[:160].replace("\n", " ")  # 본문 앞부분만 잘라 비교합니다.
        print(f"{i}. p.{page} | {snippet}")
        print("-" * 50)

original_results = retriever.invoke(query)  # 원래 질문으로 검색한 결과입니다.
print_search_preview("재작성 전 검색 결과", original_results)

In [ ]:
rewritten_results = retriever.invoke(new_query)  # 재작성한 질문으로 검색한 결과입니다.
print_search_preview("재작성 후 검색 결과", rewritten_results)

질문을 다듬으면 검색 결과가 달라질 수 있습니다.

원하는 청크를 잘 못 찾았다면, 질문을 다시 써서 한 번 더 검색해보는 것이 도움이 됩니다.


### 실습 문제 2: 1일차 종합 복습 - 질문 재작성 RAG 답변 만들기

아래 문제는 앞에서 만든 `retriever`, `llm`, `format_docs`, `rewrite_query`, `domain_rewrite_prompt`를 다시 사용합니다.

목표는 짧은 사용자 질문을 검색용 질문으로 바꾸고, 검색된 문서만 근거로 답하는 작은 RAG 답변기를 직접 완성하는 것입니다.

채울 흐름은 네 단계입니다.

1. 질문 재작성: `rewrite_query(...)`
2. 문서 검색: `retriever.invoke(...)`
3. 문맥 변환: `format_docs(...)`
4. 답변 생성: `prompt | llm | StrOutputParser()`

힌트:
- 검색에는 재작성된 질문을 넣습니다.
- 답변 프롬프트에는 원래 질문과 검색 문맥을 함께 넣습니다.
- 답변은 문서에 있는 내용만 사용하게 제한합니다.

확인할 점:
- `search_query`에 문서 공식 표현이 들어갔는가?
- 검색 결과와 답변 내용이 서로 연결되는가?
- 답변이 문서 밖 일반론으로 새지 않는가?


In [ ]:
### 아래 가이드를 참고해 직접 완성해보세요

from langchain_core.prompts import ChatPromptTemplate  # 문서 근거를 넣을 프롬프트를 만듭니다.
from langchain_core.output_parsers import StrOutputParser  # 모델 응답에서 텍스트만 꺼냅니다.

# 앞에서 만든 retriever, llm, format_docs, rewrite_query, domain_rewrite_prompt를 사용합니다.
user_question = "스마트폰으로 금융투자 거래를 할 때 비밀번호 관리에서 주의할 점은?"

# 1. 사용자 질문을 문서 검색에 더 잘 맞는 표현으로 바꾸세요.
search_query = None

# 2. 검색용 질문으로 관련 문서 청크를 찾으세요.
retrieved_docs = None

# 3. 검색 결과를 프롬프트에 넣을 문맥 문자열로 바꾸세요.
context = None

# 검색 결과가 어떤 문서를 가져왔는지 먼저 확인하세요.
print_search_preview("복습 문제 검색 결과", retrieved_docs)

# 4. 문서 근거만 사용하도록 프롬프트를 구성하세요.
review_prompt = ChatPromptTemplate.from_messages([...])

# 5. 프롬프트 -> 모델 -> 출력 파서 순서로 체인을 완성하세요.
review_chain = None

# 6. 원래 질문과 검색 문맥을 넣어 답변을 생성하세요.
answer = review_chain.invoke(None)

print("원래 질문:", user_question)
print("검색용 질문:", search_query)
print("\n답변:")
print(answer)


<details>
<summary>정답 보기</summary>

```python
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

user_question = "스마트폰으로 금융투자 거래를 할 때 비밀번호 관리에서 주의할 점은?"

# 1. 사용자 질문을 문서 검색에 더 잘 맞는 표현으로 바꿉니다.
search_query = rewrite_query(domain_rewrite_prompt, user_question)

# 2. 검색용 질문으로 관련 문서 청크를 찾습니다.
retrieved_docs = retriever.invoke(search_query)

# 3. 검색 결과를 프롬프트에 넣을 문맥 문자열로 바꿉니다.
context = format_docs(retrieved_docs)

print_search_preview("복습 문제 검색 결과", retrieved_docs)

# 4. 문서 근거만 사용하도록 프롬프트를 구성합니다.
review_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "너는 금융투자 안내 문서에 근거해 답하는 AI 어시스턴트야. "
        "반드시 제공된 문서 근거만 사용하고, 문서에 없으면 '제공된 문서 내용에서는 찾을 수 없습니다'라고 답해. "
        "답변은 핵심만 3개 이내 불릿으로 정리해."
    ),
    (
        "user",
        "질문: {question}\n\n"
        "문서 근거:\n"
        "--------------------\n"
        "{context}\n"
        "--------------------\n"
        "문서 근거만 사용해 답해줘."
    ),
])

# 5. 프롬프트 -> 모델 -> 출력 파서 순서로 체인을 완성합니다.
review_chain = review_prompt | llm | StrOutputParser()

# 6. 원래 질문과 검색 문맥을 넣어 답변을 생성합니다.
answer = review_chain.invoke({"question": user_question, "context": context})

print("원래 질문:", user_question)
print("검색용 질문:", search_query)
print("\n답변:")
print(answer)
```
</details>
